# 01 — Problem, data, and key findings

## What are we trying to solve?

For every tactical frame in the full match, assign `H`, `A`, `D`, or `X`. The hard part is timing: a nearby player may only brush the ball, a pass may be in flight, the ball may be missing, or play may have stopped before the next restart.

This notebook records only the observations that define those problems. It does not train a model or use the public labels for prediction.

## What the supplied data gives us

- Tactical ATD: ball and temporary player-track coordinates, speeds, timestamps, and frame IDs.
- Homography: the camera-to-pitch mapping, useful for visual checks but not a possession label.
- Smart Tagging: broad phase context such as restarts and goals; its intervals are not frame-level ground truth.
- The scored windows cover the full-match timeline, so availability must be checked on graded frames rather than only in the public five-minute slice.

In [ ]:
# Load the supplied files and establish one zero-based tactical frame index.
import os
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_CANDIDATES = []
if os.environ.get('MTAH_DATA_DIR'):
    DATA_CANDIDATES.append(Path(os.environ['MTAH_DATA_DIR']))
DATA_CANDIDATES.extend([ROOT / 'data', ROOT.parent / 'data'])
DATA = next((p.resolve() for p in DATA_CANDIDATES if (p / 'tactical').exists()), None)
if DATA is None:
    raise FileNotFoundError('Set MTAH_DATA_DIR to the folder containing tactical/')
os.environ['MTAH_DATA_DIR'] = str(DATA)
sys.path.insert(0, str(ROOT))

from possession_lab.baseline_pipeline import load_supplied_frame_table, graded_mask
from starter.load_smart_tagging import load as load_smart_tagging

atd, atd_report = load_supplied_frame_table(data_dir=DATA)
atd_frame = atd.copy()
atd_frame['frame'] = atd_frame['frame_id'] - 1
events = load_smart_tagging()
graded = graded_mask(len(atd_frame))
print(f'full tactical table: {len(atd_frame):,} frames')
print(f'graded frames: {graded.sum():,}; outside graded windows: {(~graded).sum():,}')
print(f'Smart Tag events: {len(events)}')

## Finding 1 — the ball is not continuously observed

The ball coordinate is a measurement with gaps, not a reliable state label. A gap can mean occlusion, an aerial pass, a player or camera obstruction, or out of play. The model therefore needs continuity, direction, receiver arrival, and restart evidence before changing ownership.

In [ ]:
ball_seen = atd_frame[['ball_x', 'ball_y']].notna().all(axis=1)
graded_ball_seen = ball_seen[graded]
print(f'ball missing on graded frames: {1 - graded_ball_seen.mean():.1%}')

seen = ball_seen.to_numpy() & graded
edges = np.diff(np.r_[0, (~seen).astype(int), 0])
starts = np.flatnonzero(edges == 1)
ends = np.flatnonzero(edges == -1)
gap_seconds = (ends - starts) / 25
if len(gap_seconds):
    print(f'graded ball gaps: {len(gap_seconds):,}')
    print(f'median gap: {np.median(gap_seconds):.2f}s; 90th percentile: {np.percentile(gap_seconds, 90):.1f}s; longest: {gap_seconds.max():.1f}s')

player_xy = [c for c in atd_frame.columns if str(c).endswith('_x') and 'ball' not in str(c).lower()]
print(f'median visible player x-coordinates: {atd_frame.loc[graded, player_xy].notna().sum(axis=1).median():.0f}')

## Finding 2 — phase tags are context, not labels

A restart or goal tag opens a review window, but the tagged interval can include preparation, the stoppage, the restart, and early live play. We use physical ball activation and stable control to locate the boundary.

In [ ]:
tag_codes = pd.Series([event.get('code') for event in events]).value_counts()
print(tag_codes.to_string())

gt_path = ROOT / 'evaluation' / 'gt' / 'BAR-ATH_possession_gt_public.csv'
if gt_path.exists():
    public_gt = pd.read_csv(gt_path)
    public = public_gt['frame'].between(11_185, 18_684)
    set_piece = np.zeros(len(public_gt), dtype=bool)
    for event in events:
        if event.get('code') == 'SET PIECES':
            left = max(0, int(event['start_frame']))
            right = min(len(public_gt) - 1, int(event['end_frame']))
            set_piece[left:right + 1] = True
    selected = set_piece & public.to_numpy()
    if selected.any():
        print('public SET PIECES label mix:', public_gt.loc[selected, 'label'].value_counts().to_dict())
        print(f'D share inside SET PIECES: {public_gt.loc[selected, "label"].eq("D").mean():.1%}')
else:
    print('public labels unavailable; tag context can still be inspected')

In [ ]:
# One compact audit plot: ball availability per minute in each graded half.
import matplotlib.pyplot as plt
from possession_lab.baseline_pipeline import add_match_time_columns

time_index = add_match_time_columns(atd_frame[['frame']].copy())
audit = pd.DataFrame({'frame': atd_frame['frame'], 'half': time_index['match_half'], 'ball_seen': ball_seen})
fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharey=True)
for ax, half in zip(axes, ('1H', '2H')):
    selected = audit[(audit['half'] == half) & graded]
    minute = ((selected['frame'] - selected['frame'].min()) // (25 * 60))
    availability = selected['ball_seen'].groupby(minute).mean()
    ax.bar(availability.index, availability.values, color='#4c78a8')
    ax.set_title(f'{half}: share of graded frames with a tracked ball')
    ax.set_ylim(0, 1)
axes[-1].set_xlabel('minute of half')
axes[0].set_ylabel('tracked share')
axes[1].set_ylabel('tracked share')
plt.tight_layout()

## Abstraction for the rest of the pipeline

The data does not directly reveal possession. It gives noisy observations of ball motion, player arrival, pressure, visibility, and football phase. The downstream notebooks therefore: (1) derive physical features, (2) combine them into scenarios, (3) establish conservative control anchors, (4) test competing transition hypotheses over intervals, and (5) release `H`, `A`, or `D` only when the evidence is temporally consistent.